In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script: Batch Quality Analysis and Production Summary
# Purpose: Analyze batch quality, compute pass/fail status, product-wise batch stats, and batch-wise summary
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: Reads 'purgo_databricks.purgo_playground.batch_qc', computes quality status, aggregates product-wise and batch-wise metrics, validates schema and data, and outputs three DataFrames for downstream use.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import StringType, DoubleType, LongType, DateType  

# -- CTE: Read and validate source table, enforce schema and data types
def read_and_validate_batch_qc():
    """
    Reads the batch_qc table, validates schema and data types, and returns a DataFrame.
    Args:
        None
    Returns:
        DataFrame: Validated batch_qc DataFrame
    Raises:
        Exception: If schema mismatch, missing required fields, or type mismatch detected
    """
    try:
        df = spark.read.table("purgo_databricks.purgo_playground.batch_qc")
    except Exception as e:
        raise Exception(f"Error reading table: {e}")
    # Validate column names and types (excluding nullable)
    expected_schema = [
        ("batch_id", StringType),
        ("product_name", StringType),
        ("production_date", DateType),
        ("quantity", LongType),
        ("quality_check_score", DoubleType)
    ]
    actual_schema = [(field.name, type(field.dataType)) for field in df.schema.fields]
    for idx, (col_name, col_type) in enumerate(expected_schema):
        if idx >= len(actual_schema):
            raise Exception(f"Column count mismatch: expected {len(expected_schema)}, got {len(actual_schema)}")
        actual_col_name, actual_col_type = actual_schema[idx]
        if actual_col_name != col_name:
            raise Exception(f"Column name mismatch at position {idx}: expected {col_name}, got {actual_col_name}")
        if actual_col_type != col_type:
            raise Exception(f"Type mismatch in field: {col_name} (expected {col_type.__name__}, got {actual_col_type.__name__})")
    # Validate required fields are present and not null
    required_fields = ["batch_id", "product_name", "quality_check_score"]
    for field in required_fields:
        if field not in df.columns:
            raise Exception(f"Missing required field: {field}")
        null_count = df.filter(F.col(field).isNull()).count()
        if null_count > 0:
            raise Exception(f"Missing required field: {field} (null values detected)")
    # Validate quality_check_score values (null, negative, NaN)
    invalid_score_df = df.filter(
        (F.col("quality_check_score").isNull()) |
        (F.col("quality_check_score") < 0) |
        (F.isnan(F.col("quality_check_score")))
    )
    invalid_rows = invalid_score_df.select("batch_id", "quality_check_score").collect()
    for row in invalid_rows:
        raise Exception(f"Invalid quality_check_score: {row['quality_check_score']} in batch_id {row['batch_id']}")
    # Detect duplicate batch_id
    dup_df = df.groupBy("batch_id").count().filter(F.col("count") > 1)
    dup_ids = [row["batch_id"] for row in dup_df.collect()]
    if dup_ids:
        raise Exception(f"Duplicate batch_id detected: {', '.join(dup_ids)}")
    return df

# -- CTE: Compute quality_status column for each batch
def compute_quality_status(df):
    """
    Adds a 'quality_status' column to the DataFrame based on quality_check_score.
    Args:
        df (DataFrame): Input DataFrame
    Returns:
        DataFrame: DataFrame with 'quality_status' column
    """
    return df.withColumn(
        "quality_status",
        F.when(F.col("quality_check_score") < 97, F.lit("fail")).otherwise(F.lit("pass"))
    )

# -- CTE: Exclude batches with null or empty product_name for product-wise aggregation
def filter_valid_product_name(df):
    """
    Filters out batches with null or empty product_name for product-wise aggregation.
    Args:
        df (DataFrame): Input DataFrame
    Returns:
        DataFrame: Filtered DataFrame
    """
    null_or_empty_df = df.filter(
        (F.col("product_name").isNull()) | (F.trim(F.col("product_name")) == "")
    )
    for row in null_or_empty_df.select("batch_id").collect():
        print(f"Error: Null or empty product_name in batch_id {row['batch_id']}")
    return df.filter(
        (F.col("product_name").isNotNull()) & (F.trim(F.col("product_name")) != "")
    )

# -- CTE: Product-wise batch analysis aggregation
def product_wise_analysis(df):
    """
    Aggregates product-wise batch analysis: total, passed, failed batches, and percentage passed.
    Args:
        df (DataFrame): Input DataFrame with 'quality_status'
    Returns:
        DataFrame: Product-wise analysis DataFrame
    """
    grouped = df.groupBy("product_name").agg(
        F.count("batch_id").alias("total_batches"),
        F.sum(F.when(F.col("quality_status") == "pass", 1).otherwise(0)).alias("passed_batches"),
        F.sum(F.when(F.col("quality_status") == "fail", 1).otherwise(0)).alias("failed_batches")
    )
    result = grouped.withColumn(
        "percentage_passed",
        F.round(F.col("passed_batches") / F.col("total_batches") * 100, 2)
    )
    return result.select(
        "product_name", "total_batches", "passed_batches", "failed_batches", "percentage_passed"
    )

# -- CTE: Batch-wise quality status selection
def batch_wise_quality_status(df):
    """
    Selects batch_id, product_name, quality_check_score, and quality_status for each batch.
    Args:
        df (DataFrame): Input DataFrame with 'quality_status'
    Returns:
        DataFrame: Batch-wise quality status DataFrame
    """
    return df.select("batch_id", "product_name", "quality_check_score", "quality_status")

# -- CTE: Batch-wise summary of production
def batch_wise_summary(df):
    """
    Aggregates total batches, total passed batches, and total failed batches.
    Args:
        df (DataFrame): Input DataFrame with 'quality_status'
    Returns:
        DataFrame: Batch-wise summary DataFrame
    """
    agg = df.agg(
        F.count("batch_id").alias("total_batches"),
        F.sum(F.when(F.col("quality_status") == "pass", 1).otherwise(0)).alias("total_passed_batches"),
        F.sum(F.when(F.col("quality_status") == "fail", 1).otherwise(0)).alias("total_failed_batches")
    )
    return agg.select("total_batches", "total_passed_batches", "total_failed_batches")

# -- Main execution: Read, transform, aggregate, and output results
try:
    # Read and validate source table
    batch_qc_df = read_and_validate_batch_qc()
    # Compute quality_status for all batches
    batch_qc_status_df = compute_quality_status(batch_qc_df)
    # Product-wise analysis (exclude null/empty product_name)
    batch_qc_valid_product_df = filter_valid_product_name(batch_qc_status_df)
    product_wise_analysis_df = product_wise_analysis(batch_qc_valid_product_df)
    # Batch-wise quality status
    batch_wise_quality_status_df = batch_wise_quality_status(batch_qc_status_df)
    # Batch-wise summary
    batch_wise_summary_df = batch_wise_summary(batch_qc_status_df)
except Exception as e:
    print(f"Error in batch quality analysis: {e}")

# -- End of script
